# Import the O*NET CSVs into MySQL

*Written 5 Nov 2026.*

The MySQL Workbench Table Data Import Wizard is slow on the large O*NET files (31,821 to 54,269 rows) and lost or repeated rows when it was stopped or run twice. This notebook imports all five cleaned O*NET CSVs into their staging tables in one run, then checks that every table has exactly as many rows as its CSV.

| CSV in `data/cleaned/` | Staging table | Rows |
|---|---|---:|
| `occupations_clean.csv` | `stg_occupations` | 1,016 |
| `software_skills_clean.csv` | `stg_software_skills` | 31,821 |
| `job_titles_clean.csv` | `stg_job_titles` | 54,269 |
| `essential_skills_clean.csv` | `stg_essential_skills` | 18,200 |
| `transferable_skills_clean.csv` | `stg_transferable_skills` | 45,500 |

**Before running:** create the tables by running `sql/JobAvenir_01_Create_Database_MySQL.sql` in MySQL Workbench, and make sure your MySQL server (or MAMP) is running.

**After running:** import the two Jobicy CSVs with `importing_jobicy_data_into_MySql.ipynb`, then run `sql/JobAvenir_02_Load_and_Analyze_MySQL.sql`.

Clear this notebook's outputs before committing it.

Step 1: Settings

In [ ]:
from pathlib import Path

# 1 = empty each staging table first, then import (use this to fix a bad or repeated import)
# 0 = only import into staging tables that are still empty
REPLACE_STAGING = 1

# MySQL connection. MYSQL_PORT must match your MySQL Workbench connection.
# The default is 3306 (standard MySQL). If your server uses another port
# (MAMP's default is 8889), put the line MYSQL_PORT=8889 in a file called .env
# in the project folder. .env is not committed to GitHub, so each teammate
# keeps their own port without editing this notebook.
MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_DATABASE = "jobavenir_db"

# Rows sent to MySQL in one batch (small enough for MySQL's default packet size)
BATCH_SIZE = 2000

# Cleaned CSVs, relative to this notebook (notebooks/)
CLEANED_FOLDER = Path("..") / "data" / "cleaned"

# Each CSV and the staging table it goes into
IMPORTS = [
    ["occupations_clean.csv", "stg_occupations"],
    ["software_skills_clean.csv", "stg_software_skills"],
    ["job_titles_clean.csv", "stg_job_titles"],
    ["essential_skills_clean.csv", "stg_essential_skills"],
    ["transferable_skills_clean.csv", "stg_transferable_skills"],
]

env_file = Path("..") / ".env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.strip().startswith("MYSQL_PORT="):
            MYSQL_PORT = int(line.split("=", 1)[1].strip())

print("Using MySQL port", MYSQL_PORT)

Step 2: Connect to MySQL

In [ ]:
import mysql.connector
from getpass import getpass

connection = mysql.connector.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=getpass("Enter your MySQL password: "),
    database=MYSQL_DATABASE
)

print("Connected to MySQL successfully!")

Step 3: Import the five O*NET CSVs

- Every column is read as text, so codes such as `15-2051.00` and source `08` keep their exact form. Blank cells stay blank (`''`), just as the Import Wizard stores them; File 2 turns blanks into NULL when it loads the core tables.
- The CSV columns must match the staging table's columns in name and order; otherwise the import stops before changing anything.
- Each table is emptied and filled inside one transaction, so a failed import leaves the table as it was.

In [ ]:
import pandas as pd

cursor = connection.cursor()
csv_row_counts = {}

for csv_name, table_name in IMPORTS:
    file_path = CLEANED_FOLDER / csv_name

    # utf-8-sig also reads a file that starts with a byte-order mark
    df = pd.read_csv(file_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    csv_row_counts[table_name] = len(df)

    # The CSV columns must match the staging table's columns, in order
    cursor.execute("SHOW COLUMNS FROM " + table_name)
    table_columns = []
    for row in cursor.fetchall():
        table_columns.append(row[0])

    if list(df.columns) != table_columns:
        raise ValueError(
            csv_name + " columns " + str(list(df.columns))
            + " do not match " + table_name + " columns " + str(table_columns)
        )

    cursor.execute("SELECT COUNT(*) FROM " + table_name)
    existing_rows = cursor.fetchone()[0]

    if existing_rows > 0 and REPLACE_STAGING == 0:
        print(table_name + ": skipped, it already has", existing_rows, "rows (REPLACE_STAGING = 0)")
        continue

    # Column names in backticks, because stg_job_titles has a column called `source`
    quoted_columns = []
    for column in table_columns:
        quoted_columns.append("`" + column + "`")

    insert_query = (
        "INSERT INTO " + table_name + " (" + ", ".join(quoted_columns) + ")"
        + " VALUES (" + ", ".join(["%s"] * len(table_columns)) + ")"
    )

    records = list(df.itertuples(index=False, name=None))

    try:
        if existing_rows > 0:
            # DELETE (not TRUNCATE) so a failed import rolls back to the old rows
            cursor.execute("DELETE FROM " + table_name)

        for start in range(0, len(records), BATCH_SIZE):
            cursor.executemany(insert_query, records[start:start + BATCH_SIZE])

        connection.commit()

    except Exception:
        connection.rollback()
        print(table_name + ": import failed, table left unchanged")
        raise

    cursor.execute("SELECT COUNT(*) FROM " + table_name)
    loaded_rows = cursor.fetchone()[0]
    print(table_name + ":", loaded_rows, "rows imported (CSV has", len(df), "rows; replaced", existing_rows, "old rows)")

Step 4: Check every staging table

Each O*NET table should match its CSV exactly. The two Jobicy tables stay at 0 until `importing_jobicy_data_into_MySql.ipynb` has been run (expected then: 74 and 167).

In [ ]:
staging_tables = [
    "stg_occupations",
    "stg_software_skills",
    "stg_job_titles",
    "stg_essential_skills",
    "stg_transferable_skills",
    "stg_jobicy_postings",
    "stg_job_skills",
]

all_match = True

for table_name in staging_tables:
    cursor.execute("SELECT COUNT(*) FROM " + table_name)
    rows_in_mysql = cursor.fetchone()[0]

    if table_name in csv_row_counts:
        rows_in_csv = csv_row_counts[table_name]
        if rows_in_mysql == rows_in_csv:
            status = "OK"
        else:
            status = "MISMATCH"
            all_match = False
        print(f"{table_name:26s} {rows_in_mysql:>7,d}   CSV {rows_in_csv:>7,d}   {status}")
    else:
        print(f"{table_name:26s} {rows_in_mysql:>7,d}   (Jobicy notebook)")

if all_match:
    print("\nAll five O*NET staging tables match their CSV files.")
else:
    print("\nAt least one table does not match its CSV: re-run Step 3 with REPLACE_STAGING = 1.")

cursor.close()
connection.close()